### Step 1: Create the existing dimension data

In [0]:
customer_data = [
    ("C1001","Raj",'Chennai'),
    ("C1002","Priya","Madurai"),
    ("C1003","Karun","Coimbatore")
]

customer_df = spark.createDataFrame(
    customer_data,
    ['customer_id','customer_name','city']
)

customer_df.show()

+-----------+-------------+----------+
|customer_id|customer_name|      city|
+-----------+-------------+----------+
|      C1001|          Raj|   Chennai|
|      C1002|        Priya|   Madurai|
|      C1003|        Karun|Coimbatore|
+-----------+-------------+----------+



### Step 2: Create the latest customer data

In [0]:
new_customer_data = [
    ('C1001','Raj','Bengaluru')
]

new_customer_df = spark.createDataFrame(
    new_customer_data,
    ['customer_id','customer_name','city']
)

new_customer_df.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|      C1001|          Raj|Bengaluru|
+-----------+-------------+---------+



### Step 3: Store the dimension as a Delta table

In [0]:
customer_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.dim_customerb")

### Step 4: Load the Delta table

In [0]:
from delta.tables import DeltaTable

customer_target = DeltaTable.forName(
    spark,
    'workspace.ecommerce_project.dim_customer'
)

### Step 5: Apply SCD Type 1

In [0]:
(
    customer_target.alias("target")
    .merge(
        new_customer_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Verify the Result

In [0]:
spark.sql( '''
          select * from workspace.ecommerce_project.dim_customer
          order by customer_id
          ''').show()

+-----------+-------------+----------+
|customer_id|customer_name|      city|
+-----------+-------------+----------+
|      C1001|          Raj| Bengaluru|
|      C1002|        Priya|   Madurai|
|      C1003|        Karun|Coimbatore|
+-----------+-------------+----------+

